# Feature engineering

**Philosophy:** Build **team-level** stats per (Season, TeamID) from regular-season data (win rate, PPG, PAPG, efficiency, seed, etc.). Then in the matchup layer we use **differences** (Team A stat minus Team B stat)—never raw team stats—to keep features comparable and avoid leakage.

Reloading feature module

In [3]:
import sys
sys.path.insert(0, '..')
import importlib
import src.features

importlib.reload(src.features)



<module 'src.features' from '/Users/calebrh/model-madness/march-machine-learning-mania-2026/notebooks/../src/features.py'>

Loading Data

In [4]:
from src.data_loading import load_regular_season_results

games = load_regular_season_results(compact=False)

In [5]:
from src import features

# Stub: build_team_season_stats expects DataFrames from data_loading
# Expected output columns: Season, TeamID, win_pct, ppg, papg, ... (see features.get_feature_columns)
try:
    features.build_team_season_stats(None)
except NotImplementedError:
    print("build_team_season_stats is a stub; implement with regular_season_results, seeds, optional ratings.")
try:
    features.get_feature_columns()
except NotImplementedError:
    print("get_feature_columns will return the list of feature names for matchup diffs.")

build_team_season_stats is a stub; implement with regular_season_results, seeds, optional ratings.
get_feature_columns will return the list of feature names for matchup diffs.


Building the

Adding winner and loser rows

In [6]:
from src.game_results import make_long_regular_season_results

team_games = make_long_regular_season_results(games)
print("Long-format regular-season rows:", team_games.shape)

team_games.head()


Long-format regular-season rows: (249058, 33)


,Season,DayNum,TeamID,OppTeamID,win,points_for,points_against,fgm,fga,fgm3,...,opp_fga3,opp_ftm,opp_fta,opp_or,opp_dr,opp_ast,opp_to,opp_stl,opp_blk,opp_pf
0,2003,10,1104,1328,1,68,62,27,58,3,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,1393,1,70,63,26,62,8,...,24,9,20,20,25,7,12,8,6,16
2,2003,10,1328,1104,0,62,68,22,53,2,...,14,11,18,14,24,13,23,7,1,22
3,2003,10,1393,1272,0,63,70,24,67,6,...,20,10,19,15,28,16,13,4,4,18
4,2003,11,1186,1458,0,55,81,20,46,3,...,12,23,27,12,24,12,9,9,3,18


Sanity check for winner and loser row aggregations

In [7]:
team_games.shape
team_games.head()

,Season,DayNum,TeamID,OppTeamID,win,points_for,points_against,fgm,fga,fgm3,...,opp_fga3,opp_ftm,opp_fta,opp_or,opp_dr,opp_ast,opp_to,opp_stl,opp_blk,opp_pf
0,2003,10,1104,1328,1,68,62,27,58,3,...,10,16,22,10,22,8,18,9,2,20
1,2003,10,1272,1393,1,70,63,26,62,8,...,24,9,20,20,25,7,12,8,6,16
2,2003,10,1328,1104,0,62,68,22,53,2,...,14,11,18,14,24,13,23,7,1,22
3,2003,10,1393,1272,0,63,70,24,67,6,...,20,10,19,15,28,16,13,4,4,18
4,2003,11,1186,1458,0,55,81,20,46,3,...,12,23,27,12,24,12,9,9,3,18


Building advanced features for the team games (can go multiple different directions from here)

In [8]:
from src.features import build_advanced_features

advanced_features = build_advanced_features(team_games)
advanced_features.head()

,Season,DayNum,TeamID,OppTeamID,win,points_for,points_against,fgm,fga,fgm3,...,opp_pf,margin,possessions,efg_pct,tov_pct,orb_pct,ftr,off_eff,def_eff,net_eff
0,2003,10,1104,1328,1,68,62,27,58,3,...,20,6,73.5000,0.491379,0.312925,0.388889,0.310345,92.517007,84.353741,8.163265
1,2003,10,1272,1393,1,70,63,26,62,8,...,16,7,68.7625,0.483871,0.189057,0.375000,0.306452,101.799673,91.619706,10.179967
2,2003,10,1328,1104,0,62,68,22,53,2,...,22,-6,73.5000,0.433962,0.244898,0.294118,0.415094,84.353741,92.517007,-8.163265
3,2003,10,1393,1272,0,63,70,24,67,6,...,18,-7,68.7625,0.402985,0.174514,0.416667,0.298507,91.619706,101.799673,-10.179967
4,2003,11,1186,1458,0,55,81,20,46,3,...,18,-26,66.9500,0.467391,0.283794,0.200000,0.369565,82.150859,120.985810,-38.834951


Getting season long regular season stats

In [9]:
from src.features import build_regular_season_team_stats

season_features = build_regular_season_team_stats(advanced_features)
season_features.head()

,Season,TeamID,games_played,win_pct,avg_margin,avg_points_for,avg_points_against,off_eff,def_eff,net_eff,efg_pct,tov_pct,orb_pct,ftr
0,2003,1102,28,0.428571,0.250000,57.250000,57.000000,103.835749,103.601436,0.234313,0.584407,0.205580,0.168235,0.446693
1,2003,1103,27,0.481481,0.629630,78.777778,78.148148,110.667469,110.408760,0.258709,0.536564,0.179502,0.305803,0.465135
2,2003,1104,28,0.607143,4.285714,69.285714,65.000000,103.557697,97.794103,5.763594,0.475785,0.199612,0.371256,0.372350
3,2003,1105,26,0.269231,-4.884615,71.769231,76.653846,93.616492,100.207433,-6.590941,0.457983,0.242292,0.335166,0.359501
4,2003,1106,28,0.464286,-0.142857,63.607143,63.750000,93.773763,94.224204,-0.450441,0.481697,0.251113,0.349480,0.307563


Sanity check on season features

In [10]:
season_features.groupby(["Season", "TeamID"]).size().value_counts()
season_features.isna().sum()
season_features.describe()
season_features.sort_values("net_eff", ascending = False).head(20)

,Season,TeamID,games_played,win_pct,avg_margin,avg_points_for,avg_points_against,off_eff,def_eff,net_eff,efg_pct,tov_pct,orb_pct,ftr
5585,2019,1211,33,0.909091,23.787879,88.848485,65.060606,123.829359,90.759667,33.069693,0.594681,0.144751,0.301536,0.359958
4883,2017,1211,33,0.969697,23.424242,84.575758,61.151515,119.293332,86.274725,33.018607,0.580935,0.158583,0.291260,0.394115
4215,2015,1246,34,1.000000,20.941176,74.911765,53.970588,115.910391,83.659554,32.250837,0.520134,0.163519,0.402745,0.451499
7691,2025,1181,34,0.911765,20.794118,82.705882,61.911765,122.498919,91.694300,30.804619,0.574671,0.136420,0.335700,0.327781
3875,2014,1257,34,0.852941,21.147059,82.117647,60.970588,118.661179,88.063803,30.597376,0.540508,0.148240,0.370055,0.414850
6286,2021,1211,26,1.000000,23.000000,92.115385,69.115385,120.937119,90.584286,30.352834,0.610733,0.156097,0.271928,0.371953
6638,2022,1211,29,0.896552,22.482759,87.827586,65.344828,117.564555,87.544926,30.019629,0.594659,0.155430,0.269913,0.300106
3467,2013,1196,33,0.787879,17.939394,71.636364,53.696970,114.435507,85.672215,28.763292,0.558752,0.176417,0.338462,0.299035
1783,2008,1242,33,0.909091,19.515152,81.181818,61.666667,117.482390,89.292836,28.189554,0.561901,0.186586,0.379112,0.383291
7006,2023,1222,34,0.911765,18.529412,75.029412,56.500000,114.118478,85.987843,28.130635,0.526515,0.142771,0.368546,0.290498


Building recent features (stats for teams in x recent games (default 10))

In [11]:
from src.features import get_recent_features

recent_features = get_recent_features(team_games)
recent_features.head()

,Season,TeamID,lastx_win_pct,lastx_avg_margin,lastx_off_eff,lastx_def_eff,lastx_net_eff
0,2003,1102,0.2,-8.0,100.049778,114.969991,-14.920213
1,2003,1103,0.5,0.6,110.346843,109.392363,0.954480
2,2003,1104,0.4,1.1,107.131388,105.967896,1.163492
3,2003,1105,0.3,-0.2,99.156553,99.337066,-0.180513
4,2003,1106,0.4,0.1,92.946370,93.041995,-0.095626


Combine recent and season features

In [12]:
from src.features import combine_features

combined_features = combine_features(season_features, recent_features)
combined_features.head()

,Season,TeamID,games_played,win_pct,avg_margin,avg_points_for,avg_points_against,off_eff,def_eff,net_eff,efg_pct,tov_pct,orb_pct,ftr,lastx_win_pct,lastx_avg_margin,lastx_off_eff,lastx_def_eff,lastx_net_eff
0,2003,1102,28,0.428571,0.250000,57.250000,57.000000,103.835749,103.601436,0.234313,0.584407,0.205580,0.168235,0.446693,0.2,-8.0,100.049778,114.969991,-14.920213
1,2003,1103,27,0.481481,0.629630,78.777778,78.148148,110.667469,110.408760,0.258709,0.536564,0.179502,0.305803,0.465135,0.5,0.6,110.346843,109.392363,0.954480
2,2003,1104,28,0.607143,4.285714,69.285714,65.000000,103.557697,97.794103,5.763594,0.475785,0.199612,0.371256,0.372350,0.4,1.1,107.131388,105.967896,1.163492
3,2003,1105,26,0.269231,-4.884615,71.769231,76.653846,93.616492,100.207433,-6.590941,0.457983,0.242292,0.335166,0.359501,0.3,-0.2,99.156553,99.337066,-0.180513
4,2003,1106,28,0.464286,-0.142857,63.607143,63.750000,93.773763,94.224204,-0.450441,0.481697,0.251113,0.349480,0.307563,0.4,0.1,92.946370,93.041995,-0.095626
